# Can we predict `passed_quiz`?
Protocol: `protocol.yaml` (group holdout on `student_key` from `holdout.csv`, 5-fold GroupKFold CV on training rows, choose by mean CV ROC AUC, score the chosen model once on holdout).

Feature timing (from `columns.csv`): `enrol_year`, `school_code`, `prior_score` (placement test) are known before the session; `session_minutes` and `hints_used` are session-level measures. The task is predicting the quiz that *closes* the session, so I treat these as available at prediction time (late-session use). `student_key` is excluded (entity key).

In [ ]:
import json, hashlib, numpy as np, pandas as pd, yaml
from sklearn.base import clone
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
proto = yaml.safe_load(open('protocol.yaml'))
sha = lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
assert sha('data.csv') == proto['dataset']['sha256'] and sha('holdout.csv') == proto['split']['holdout_sha256']
df = pd.read_csv('data.csv')
assert len(df) == proto['dataset']['rows']
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(hold_idx) == proto['split']['holdout_rows']
is_hold = np.zeros(len(df), bool); is_hold[hold_idx] = True
train, hold = df[~is_hold], df[is_hold]
assert not set(train.student_key) & set(hold.student_key)
print(len(train), len(hold), train.passed_quiz.mean(), hold.passed_quiz.mean())

In [ ]:
FEATURES = ['enrol_year','school_code','prior_score','session_minutes','hints_used']
TARGET = 'passed_quiz'
# All candidates named before any is scored. Preprocessing lives inside the pipelines (fit per fold).
candidates = {
  'baseline_prior': make_pipeline(DummyClassifier(strategy='prior')),
  'logreg': make_pipeline(SimpleImputer(), StandardScaler(), LogisticRegression(C=1.0, max_iter=1000)),
  'random_forest': make_pipeline(SimpleImputer(), RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=0, n_jobs=-1)),
  'hist_gb': make_pipeline(HistGradientBoostingClassifier(learning_rate=0.05, max_iter=200, max_depth=3, min_samples_leaf=40, random_state=0)),
}
print(list(candidates))

In [ ]:
X, y, g = train[FEATURES], train[TARGET].to_numpy(), train.student_key
cv = GroupKFold(n_splits=proto['cv']['n_splits'])
folds = [(tr, te) for tr, te in cv.split(X, y, g)]
cv_scores = {}
for name, m in candidates.items():
    s = []
    for tr, te in folds:
        mm = clone(m).fit(X.iloc[tr], y[tr])
        s.append(float(roc_auc_score(y[te], mm.predict_proba(X.iloc[te])[:,1])))
    cv_scores[name] = s
cvdf = pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T
cvdf

In [ ]:
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
print('chosen by mean CV roc_auc:', chosen)
final = clone(candidates[chosen]).fit(X, y)
holdout_auc = float(roc_auc_score(hold[TARGET], final.predict_proba(hold[FEATURES])[:,1]))
print('HEADLINE holdout roc_auc (scored once):', round(holdout_auc,4))

In [ ]:
train_rows = df.index[~is_hold].to_numpy()
evidence = {
  'features': FEATURES,
  'candidates': list(candidates),
  'cv_folds': [[train_rows[tr].tolist(), train_rows[te].tolist()] for tr,te in folds],
  'cv_scores': cv_scores,
  'chosen': chosen,
  'holdout_rows': hold_idx.tolist(),
  'holdout_scores': {chosen: holdout_auc},
}
json.dump(evidence, open('evidence.json','w'))